# LangChain Expression Language (LCEL)



<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>
<b>About</b><br><br>

This notebook is derived from the following notebook, with modifications and extensions: https://github.com/AI-Engineering-bootcamp/ai-eng-nbs-public/blob/master/langchain-expression-language-202503.ipynb
</div>

<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>

## Environment Setup

<br>

> 💡 **Important:**
>
> This notebook requires **LangChain < 0.3**.
>
> Below, you will find two options for installing the required dependencies. Choose the one that best matches your environment::
>
> 🖥️ **Running locally?**  
> → Use **Option A** to create a dedicated virtual environment (recommended).
>
> ☁️ **Using Google Colab or want a quick setup?**  
> → Use **Option B** to install the required dependencies directly.
>

<br><br>


### Option A: Use a virtual environment (recommended)

To create a virtual environment, follow the steps in this document:
- https://gist.github.com/luisjunco/6ade3de82b7804f3ebf912866b4efff5

For this notebook, you'll need these dependencies:

```bash
python -m pip install \
    "langchain<0.3" \
    "langchain-core<0.3" \
    "langchain-community<0.3" \
    "langchain-openai<0.2" \
    ipykernel
```


### Option B: Install dependencies directly

If you are using Google Colab, or you're having problems configuring a virtual environment, create a code cell and run the command below:

```python
!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"
```

<br>


</div>

<br>

## Check LangChain version

For this notebook, you'll need LangChain 0.2.x (e.g., 0.2.17)

To check your LangChain version, you can run the command below:

In [5]:
!pip show langchain

Name: langchain
Version: 0.2.17
Summary: Building applications with LLMs through composability
Home-page: https://github.com/langchain-ai/langchain
Author: 
Author-email: 
License: MIT
Location: /Users/luis/Desktop/ironhack_sept26/demos_ai_engineering_sept2026/.venv/langchain-v0.2.x/lib/python3.11/site-packages
Requires: aiohttp, langchain-core, langchain-text-splitters, langsmith, numpy, pydantic, PyYAML, requests, SQLAlchemy, tenacity
Required-by: langchain-community


<br>

## Intro to LCEL



The **L**ang**C**hain **E**xpression **L**anguage (LCEL) is a abstraction of some interesting Python concepts into a format that enables a "minimalist" code layer for building chains of LangChain components.

LCEL comes with strong support for:

1. Superfast development of chains.
2. Advanced features such as streaming, async, parallel execution, and more.
3. Easy integration with LangSmith and LangServe.

## LCEL Syntax

To understand LCEL syntax let's first build a simple chain in typical Python syntax.

In [6]:
import warnings
warnings.filterwarnings("ignore")
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [7]:

prompt = ChatPromptTemplate.from_template(
    "Give me a small report about {topic}"
)

model = ChatOpenAI(
    model="gpt-6-luna",
    model_kwargs={"reasoning_effort": "none"},
    openai_api_key=OPENAI_API_KEY
)

output_parser = StrOutputParser()


In typical LangChain we would chain these together using an `LLMChain`:

In [8]:
from langchain.chains import LLMChain

chain = LLMChain(
    prompt=prompt,
    llm=model,
    output_parser=output_parser
)

# and run
out = chain.run(topic="Artificial Intelligence")
print(out)

# Artificial Intelligence: A Brief Report

**Artificial intelligence (AI)** is the field of creating computer systems that can perform tasks usually associated with human intelligence. These tasks include understanding language, recognizing images, making predictions, and solving problems.

### How AI works
AI systems learn patterns from data. Some use **machine learning**, which improves performance through examples, while **deep learning** uses layered neural networks to handle complex tasks such as speech and image recognition. Generative AI can produce new text, images, audio, and other content.

### Common uses
AI is used in virtual assistants, translation tools, medical imaging, recommendation systems, fraud detection, manufacturing, and transportation. It can help people work more efficiently and analyze large amounts of information.

### Benefits and challenges
AI can save time, support research, and automate repetitive tasks. However, it can also make mistakes, reflect bias in

<br>

Using LCEL the format is different, rather than relying on `Chains` we simple chain together each component using the pipe operator `|`:

In [9]:
lcel_chain = prompt | model | output_parser

# and run
out = lcel_chain.invoke({"topic": "Artificial Intelligence"})
print(out)

# Artificial Intelligence: A Brief Report

Artificial intelligence (AI) is the field of creating computer systems that can perform tasks commonly associated with human intelligence. These tasks include understanding language, recognizing images, making predictions, and solving problems.

## How AI works
AI systems learn patterns from data. Many modern tools use **machine learning**, in which a system improves its performance by training on examples. **Generative AI** is a type of AI that can produce new content, such as text, images, audio, or code.

## Common uses
AI is used in healthcare, education, transportation, finance, customer service, and scientific research. Examples include helping doctors analyze medical images, recommending products, translating languages, and assisting with routine office tasks.

## Benefits and challenges
AI can help people work more efficiently, analyze large amounts of information, and develop new services. However, it can also produce inaccurate resul

Pretty cool, the way that this pipe operator is being used is that it is taking output from the function to the _left_ and feeding it into the function on the _right_.

## How the Pipe Operator Works

To really understand LCEL we can take a look at how this pipe operation works. We know it takes output from the _right_ and feeds it to the _left_ — but this isn't typical Python, so how is this implemented? Let's try creating our own version of this with some simple functions.

We will be using the `__or__` method within Python class objects. When we place two classes together like `chain = class_a | class_b` the Python interpreter will check if these classes contain this `__or__` method. If they do then our `|` code will be translated into `chain = class_a.__or__(class_b)`.

That means both of these patterns will return the same thing:

```python
# object approach
chain = class_a.__or__(class_b)
chain("some input")

# pipe approach
chain = class_a | class_b
chain("some input")

```

With that in mind, we can build a `Runnable` class that consumes a function and turns it into a function that can be chained with other functions using the pipe operator `|`.

In [10]:
class Runnable:
    def __init__(self, func):
        self.func = func

    def __or__(self, other):
        def chained_func(*args, **kwargs):
            # the other func consumes the result of this func
            return other(self.func(*args, **kwargs))
        return Runnable(chained_func)

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)

Let's implement this to take the value `3`, add `5` (giving `8`), and multiply by `2` (giving `16`).

In [11]:
def add_five(x):
    return x + 5

def multiply_by_two(x):
    return x * 2

# wrap the functions with Runnable
add_five = Runnable(add_five)
multiply_by_two = Runnable(multiply_by_two)

# run them using the object approach
chain = add_five.__or__(multiply_by_two)
chain(3)  # should return 16

16

Using `__or__` directly we get the correct answer, now let's try using the pipe operator `|` to chain them together:

In [12]:
# chain the runnable functions together
chain = add_five | multiply_by_two

# invoke the chain
chain(3)  # we should return 16

16

Using either method we get the same response, at it's core this is what LCEL is doing, but there is more.

## LCEL Deep Dive

Now that we understand what this syntax is doing under the hood, let's explore it within the context of LCEL and see a few of the additional methods that LangChain has provided to maximize flexibility when working with LCEL.

### Runnables

When working with LCEL we may find that we need to modify the structure or values being passed between components — for this we can use _runnables_. Let's try:

In [13]:
!pip install docarray

  Using cached docarray-0.41.0-py3-none-any.whl.metadata (36 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached docarray-0.41.0-py3-none-any.whl (302 kB)
Using cached rich-15.0.0-py3-none-any.whl (310 kB)
Using cached markdown_it_py-4.2.0-py3-none-any.whl (91 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [14]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import DocArrayInMemorySearch

embedding = OpenAIEmbeddings(
    model="text-embedding-ada-002",
    openai_api_key=OPENAI_API_KEY
)

vecstore_a = DocArrayInMemorySearch.from_texts(
    ["half the info will be here", "James' birthday is the 7th December"],
    embedding=embedding
)
vecstore_b = DocArrayInMemorySearch.from_texts(
    ["and half here", "James was born in 1994"],
    embedding=embedding
)


First let's try passing a question through to a single one of these `vecstore` objects:

In [15]:
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough
)

retriever_a = vecstore_a.as_retriever()
retriever_b = vecstore_b.as_retriever()

prompt_str = """Answer the question below using the context:

Context: {context}

Question: {question}

Answer: """
prompt = ChatPromptTemplate.from_template(prompt_str)

retrieval = RunnableParallel(
    {"context": retriever_a, "question": RunnablePassthrough()}
)

chain = retrieval | prompt | model | output_parser

We use two new objects here, `RunnableParallel` and `RunnablePassthrough`. The `RunnableParallel` object allows us to define multiple values and operations, and run them all in parallel. Here we call `retriever_a` using the input to our chain (below), and then pass the results from `retriever_a` to the next component in the `chain` via the `"context"` parameter.

<div>
<img src="./img/lecl.webp" alt='auto' width="1000"/>
</div>

The `RunnablePassthrough` object is used as a `"passthrough"` take takes any input to the current component (`retrieval`) and allows us to provide it in the component output via the `"question"` key.

In [16]:
out = chain.invoke("when was James born?")
print(out)

James was born on 7 December.


Here we have used `RunnableParallel` to create two parallel streams of information, one for `"context"` that is information fed in from `retriever_a`, and another for `"question"` which is the _passthrough_ information, ie the information that is passed through from our `chain.invoke("when was James born?")` call.

Using this information the chain is close to answering the question but it doesn't have enough information, it is missing the information that we have stored in `retriever_b`. Fortunately, we can have multiple parallel information streams using the `RunnableParallel` object.

In [17]:
prompt_str = """Answer the question below using the context:

Context:
{context_a}
{context_b}

Question: {question}

Answer: """
prompt = ChatPromptTemplate.from_template(prompt_str)

retrieval = RunnableParallel(
    {
        "context_a": retriever_a, "context_b": retriever_b,
        "question": RunnablePassthrough()
    }
)

chain = retrieval | prompt | model | output_parser

In [18]:
out = chain.invoke("when was James born?")
print(out)

James was born on 7 December 1994.


Now the `RunnableParallel` object is allowing us to search with `retriever_a` _and_ `retriever_b` in parallel, ie at the same time.

## Runnable Lambdas

The `RunnableLambda` is a LangChain abstraction that allows us to turn Python functions into pipe-compatible function, similar to the `Runnable` class we created near the beginning of this notebook.

Let's try it out with our earlier `add_five` and `multiply_by_two` functions.

In [19]:
from langchain_core.runnables import RunnableLambda

# wrap the functions with RunnableLambda
add_five = RunnableLambda(add_five)
multiply_by_two = RunnableLambda(multiply_by_two)

As with our earlier `Runnable` abstraction, we can use `|` operators to chain together our `RunnableLambda` abstractions.

In [20]:
chain = add_five | multiply_by_two

Unlike our `Runnable` abstraction, we cannot run the `RunnableLambda` chain by calling it directly, instead we must call `chain.invoke`:

In [21]:
chain.invoke(3)

16

As before, we can see the same answer. Naturally we can feed custom functions into our chains using this approach. Let's try a short chain and see where we might want to insert a custom function:

In [22]:
prompt_str = "Tell me an short fact about {topic}"
prompt = ChatPromptTemplate.from_template(prompt_str)

chain = prompt | model | output_parser

In [23]:
chain.invoke({"topic": "Artificial Intelligence"})

'Artificial intelligence can recognize patterns in data, but it doesn’t understand them the way humans do.'

In [24]:
chain.invoke({"topic": "Artificial Intelligence"})

'AI systems can learn patterns from data, but they don’t understand the world the way humans do.'

The returned text always includes the initial `"Here's a short fact about ...\n\n"` — let's add a function to split on double newlines `"\n\n"` and only return the fact itself.

In [25]:
def extract_fact(x):
    if "\n\n" in x:
        return "\n".join(x.split("\n\n")[1:])
    else:
        return x

get_fact = RunnableLambda(extract_fact)

chain = prompt | model | output_parser | get_fact

In [26]:
chain.invoke({"topic": "Artificial Intelligence"})

'AI systems can identify patterns in data, but they don’t understand information the way humans do.'

In [27]:
chain.invoke({"topic": "Artificial Intelligence"})

'AI systems can learn patterns from data, but they don’t understand information the way humans do.'

Now we're getting well formatted outputs using our final custom `get_fact` function.

<br>

---

That covers the essentials you need to getting started and building with the **L**ang**C**hain **E**xpression **L**anguage (LCEL). With it, we can put together chains very easily.

The pros and cons of LCEL are varied. Those that love it tend to focus on the minimalist code style, LCEL's support for streaming, parallel operations, and async, and LCEL's nice integration with LangChain's focus on chaining components together.

There are also people that are less fond of LCEL. Typically people will point to it being yet another abstraction on top of an already very abstract library, that the syntax is confusing, against [the Zen of Python](https://peps.python.org/pep-0020/), and requires too much effort to learn new (or uncommon) syntax.

Both viewpoints are entirely valid, LCEL is a very different approach — ofcourse there are major pros and major cons. In either case, if you're willing to spend some time learning the syntax, it allows us to develop very quickly, and with that in mind it's well worth learning.

---